# 1 · Load OSM network consolidé (avec enrichissements intégrés)

Ce notebook construit des couches réseau marche et vélo à partir d'OpenStreetMap avec enrichissements automatiques :

1. Charger la zone d'étude (AOI) · shapefile ou géocodage.
2. Télécharger large via OSMnx `features_from_polygon` · ways + nodes.
3. Classifier marche et vélo par règles explicites.
4. Consolider la marchabilité:
   - traversées synthétiques depuis nodes `highway=crossing` (optionnel)
   - proxy trottoirs via axes routiers (optionnel)
   - bords de plateformes `public_transport=platform` (optionnel)
5. **Enrichissements intégrés** : attributs OSM nœuds + surface vélo
6. Export paramétrable (edges · nodes · formats) avec enrichissements inclus.

In [ ]:

from __future__ import annotations

import json
from pathlib import Path

import warnings
warnings.filterwarnings("ignore")
import math
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import LineString
from shapely.ops import unary_union

import osmnx as ox

pd.set_option('display.max_columns', None)  

## Explore OSM

In [ ]:
# # TEST et DIAGNOSTIC A PARTIR DE POINT

# 1) récupérer les ways autour du point, et lire highway/maxspeed/zone tags
pt = (46.094383, 5.785896)  # (lat, lon)
cand = ox.features_from_point(pt, dist=100, tags={"highway": True}).reset_index()
cand = gpd.GeoDataFrame(cand, geometry="geometry", crs="EPSG:4326")
cand = cand[cand.geometry.geom_type.isin(["LineString","MultiLineString"])].copy()

cand#.tail(4)

In [ ]:
# Plot interactif avec hover pour voir les IDs
import plotly.graph_objects as go
import plotly.express as px

# Préparer les données pour plotly
geometries = []
ids = []
highways = []
maxspeeds = []
sidewalks = []

for idx, row in cand.iterrows():
    if row.geometry is not None:
        if row.geometry.geom_type == 'LineString':
            coords = list(row.geometry.coords)
            lons, lats = zip(*coords)
            geometries.append((lons, lats))
            ids.append(str(row.get('id', 'N/A')))
            highways.append(str(row.get('highway', 'N/A')))
            maxspeeds.append(str(row.get('maxspeed', 'N/A')))
            sidewalks.append(str(row.get('sidewalk', 'N/A')))

# Créer la figure plotly
fig = go.Figure()

# Ajouter chaque ligne avec hover
for i, (geom, id_val, hw, ms, sw) in enumerate(zip(geometries, ids, highways, maxspeeds, sidewalks)):
    lons, lats = geom
    fig.add_trace(go.Scattermapbox(
        mode="lines",
        lon=lons,
        lat=lats,
        name=f"ID: {id_val}",
        hovertemplate=f"<b>ID:</b> {id_val}<br>" +
                     f"<b>Highway:</b> {hw}<br>" +
                     f"<b>Maxspeed:</b> {ms}<br>" +
                     f"<b>Sidewalk:</b> {sw}<br>" +
                     "<extra></extra>",
        line=dict(width=3, color='blue'),
        opacity=0.7
    ))

# Configuration de la carte
fig.update_layout(
    mapbox=dict(
        style="open-street-map",
        center=dict(lat=cand.geometry.centroid.y.mean(), lon=cand.geometry.centroid.x.mean()),
        zoom=14
    ),
    showlegend=False,
    height=600,
    margin=dict(l=0, r=0, t=0, b=0)
)

fig.show()

In [ ]:

cand.loc[cand.id.isin([345485306])]

## Paramètres

In [ ]:
# Sorties
output_dir = Path("../../Data/input/networkGG")
output_dir.mkdir(parents=True, exist_ok=True)

# Zone d'étude
area_mode = "shapefile"  # "shapefile" | "geocode"
shapefile_path = Path("../../Data/input/network_agreg/AGGLO_PERIMETRE_AVEC_LAC-SHP/AGGLO_PERIMETRE_AVEC_LAC.shp")
places = ["Genève, Switzerland", "Vandoeuvres, Switzerland", "Cologny, Switzerland", "Collonge-Bellerive, Switzerland", "Choulex, Switzerland"]  # si geocode

# Exports
export_nodes = True
export_edges = True
export_graphml = True
export_formats = ["parquet", "geojson"]  # "parquet" | "geojson" | "gpkg"

# CRS · séparer opérations métriques et export final
operation_crs = "EPSG:2056"  # CRS métrique pour calculs (distances, buffers, etc.)
export_crs = "EPSG:4326"     # CRS final pour les exports

# Configuration graphe vélo
bike_directed_graph = True   # True = graphe orienté respectant le sens de circulation

# Enrichissements automatiques (appliqués en section 8)
enable_enrichments = True    # True = enrichir nœuds avec attributs OSM + surface vélo
# Si activé, ajoute automatiquement :
# - Nœuds : crossing, traffic_signals, barrier, traffic_calming
# - Arêtes vélo : surface OSM brute
# - Détection zones conflit piéton-vélo

# Tenter de nettoyer les doublons, attention aux suppressions excessives!! A améliorer
clean_doublons = True

# Options consolidation marche
add_platform_edges = True
add_crossing_edges = True
add_sidewalk_proxy = True

# Proxy trottoir · règles simples
proxy_include_highways = {"residential","living_street","service","unclassified","tertiary", "path", "track"}

proxy_maxspeed_kmh = 60  # inclure si maxspeed <= seuil, si maxspeed est parsable
proxy_require_sidewalk_tag = False  # True = inclure proxy seulement si sidewalk=* présent

# Traversées synthétiques · paramètres simples
crossing_snap_dist_m = 12.0      # distance max pour rattacher une traversée à une route
crossing_halfwidth_m = 6.0       # demi-largeur supposée de chaussée (segment total = 2*halfwidth)
crossing_min_road_classes = {"residential","living_street","service","unclassified","tertiary","secondary","primary", "path", "track"}

# Nettoyage léger
deduplicate_by_geometry = True
simplify_tol_m = 0.0  # 0 = off

## Fonctions utilitaires (simples et réutilisables)

In [ ]:
def load_aoi(area_mode: str, shapefile_path: Path, places: list[str]) -> gpd.GeoDataFrame:
    """Retourne une GeoDataFrame (EPSG:4326) contenant un polygone d'AOI."""
    if area_mode == "shapefile":
        gdf = gpd.read_file(shapefile_path)
        gdf = gdf.to_crs("EPSG:4326") if gdf.crs is not None else gdf.set_crs("EPSG:4326")
        geom = unary_union(gdf.geometry.values)
        return gpd.GeoDataFrame({"name":["aoi"]}, geometry=[geom], crs="EPSG:4326")

    if area_mode == "geocode":
        polys = []
        for p in places:
            g = ox.geocode_to_gdf(p).to_crs("EPSG:4326")
            polys.append(unary_union(g.geometry.values))
        geom = unary_union(polys)
        return gpd.GeoDataFrame({"name":["aoi"]}, geometry=[geom], crs="EPSG:4326")

    raise ValueError(f"area_mode inconnu: {area_mode}")


def to_crs(gdf: gpd.GeoDataFrame, crs: str) -> gpd.GeoDataFrame:
    """Convertit vers le CRS spécifié en gérant les cas sans CRS défini."""
    if gdf.crs is None:
        return gdf.set_crs("EPSG:4326").to_crs(crs)
    return gdf.to_crs(crs)


def to_metric_crs(gdf: gpd.GeoDataFrame, metric_crs: str) -> gpd.GeoDataFrame:
    """Convertit vers un CRS métrique pour les calculs de distance."""
    return to_crs(gdf, metric_crs)


def to_export_crs(gdf: gpd.GeoDataFrame, export_crs: str) -> gpd.GeoDataFrame:
    """Convertit vers le CRS d'export final."""
    return to_crs(gdf, export_crs)


def explode_lines(gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    gdf = gdf[gdf.geometry.notna()].copy()
    return gdf.explode(index_parts=False, ignore_index=True)


def simplify_gdf(gdf: gpd.GeoDataFrame, tol_m: float, metric_crs: str) -> gpd.GeoDataFrame:
    """Simplifie en métrique puis reprojette dans le CRS d'origine."""
    if tol_m and tol_m > 0:
        original_crs = gdf.crs
        gdf_metric = to_metric_crs(gdf, metric_crs)
        gdf_metric = gdf_metric.copy()
        gdf_metric["geometry"] = gdf_metric.geometry.simplify(tol_m, preserve_topology=True)
        return to_crs(gdf_metric, str(original_crs))
    return gdf

import math
import numpy as np
from shapely.geometry import LineString

def mean_bearing(line: LineString) -> float:
    """
    Calcule une orientation moyenne simple d'une LineString (en radians).
    Approche volontairement robuste et peu coûteuse.
    """
    if line is None or line.is_empty:
        return np.nan

    coords = list(line.coords)
    if len(coords) < 2:
        return np.nan

    x0, y0 = coords[0]
    x1, y1 = coords[-1]

    dx = x1 - x0
    dy = y1 - y0

    if dx == 0 and dy == 0:
        return np.nan

    return math.atan2(dy, dx)


def safe_object_columns_for_parquet(gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    """Normalise les colonnes object mixtes (ex. osmid liste/scalar) pour éviter les erreurs parquet."""
    out = gdf.copy()
    for col in out.columns:
        if col == out.geometry.name:
            continue
        if out[col].dtype == "object":
            has_list = out[col].apply(lambda v: isinstance(v, (list, tuple, set))).any()
            if has_list:
                out[col] = out[col].apply(
                    lambda v: None if pd.isna(v) else json.dumps(list(v)) if isinstance(v, (list,tuple,set)) else json.dumps([v])
                )
            else:
                out[col] = out[col].apply(lambda v: None if pd.isna(v) else str(v))
    return out


def export_gdf(gdf: gpd.GeoDataFrame, stem: str, export_formats: list[str], output_dir: Path, export_crs: str):
    """Export multi-formats dans le CRS d'export spécifié."""
    gdf_export = to_export_crs(gdf, export_crs)
    
    if "parquet" in export_formats:
        safe_object_columns_for_parquet(gdf_export).to_parquet(output_dir / f"{stem}.parquet", index=False)
    if "geojson" in export_formats:
        gdf_export.to_file(output_dir / f"{stem}.geojson", driver="GeoJSON")
    if "gpkg" in export_formats:
        gdf_export.to_file(output_dir / f"{stem}.gpkg", driver="GPKG")

## 1 · Charger Area Of Interest

3. Union des géométries multiples si nécessaire

**Zone d'étude (Area of Interest) :**2. Conversion en EPSG:4326 (coordonnées géographiques)

- **Mode shapefile** : Charge un périmètre depuis un fichier SIG existant1. Lecture/géocodage de la zone d'étude

- **Mode geocode** : Délimite automatiquement via OpenStreetMap (ex: "Genève, Switzerland")**Processus :**


In [ ]:

aoi = load_aoi(area_mode, shapefile_path, places)
aoi_poly = aoi.geometry.iloc[0]
aoi


## 2 · Télécharger large (ways + nodes) via features_from_polygon

- Évite les biais de routabilité d'OSMnx

**Stratégie de téléchargement exhaustif :**- Permet classification personnalisée post-téléchargement

- **Ways** : Tous les segments `highway=*` (routes, chemins, pistes)- Inclut footways, paths, cycleways souvent exclus

- **Nodes** : Points de traversée `highway=crossing`- Récupère TOUS les éléments OSM, même non-routables

**Avantages vs `graph_from_polygon()` :**

In [ ]:

TAGS_WAYS = {"highway": True}
TAGS_NODES = {"highway": "crossing"}

ways_raw = ox.features_from_polygon(aoi_poly, TAGS_WAYS).reset_index()
ways_raw = ox.features_from_polygon(aoi_poly, TAGS_WAYS).reset_index()
ways_raw = gpd.GeoDataFrame(ways_raw, geometry="geometry", crs="EPSG:4326")
ways_raw = ways_raw[ways_raw.geometry.type.isin(["LineString","MultiLineString"])].copy()
ways_raw = explode_lines(ways_raw)

# ID stable OSM
ways_raw["osm_id"] = ways_raw["element"].astype(str) + "/" + ways_raw["id"].astype(str)

try:
    nodes_cross = ox.features_from_polygon(aoi_poly, TAGS_NODES).reset_index()
    nodes_cross = gpd.GeoDataFrame(nodes_cross, geometry="geometry", crs="EPSG:4326")
    nodes_cross = nodes_cross[nodes_cross.geometry.type.isin(["Point"])].copy()
except Exception:
    nodes_cross = gpd.GeoDataFrame(columns=["geometry"], geometry="geometry", crs="EPSG:4326")

ways_raw.shape, nodes_cross.shape


## 3 · Classifier marche et vélo par règles

**Logique robuste** : Gère les tags OSM incomplets ou ambigus

**Classification piéton :**
- **Dédiés** : `footway`, `pedestrian`, `steps`, `platform`
- **Chemins** : `path` (sentiers partagés)
- **Exclusions** : `foot=no`, `access=no/private`

**Classification vélo étendue :**
- **Pistes séparées** : `cycleway`, `cycleway=track/separated`
- **Bandes cyclables** : `cycleway=lane/shared_lane` (marquage au sol)
- **Chemins** : `path`, `track`, `bridleway` avec accès vélo
- **Aires spéciales** : Zones piétonnes, voies de bus avec accès vélo
- **Routes partagées** : `primary` à `service` (circulation motorisée)
- **Exclusions avancées** : Autoroutes, routes rapides sauf permission explicite

In [ ]:
# --------------------------------------------------
# Section 3 · Classification marche / vélo à partir des ways OSM
# Objectif:
# · garder un download large (toutes les ways highway pertinentes)
# · classer marche et vélo par règles
# · assurer un proxy piéton quand sidewalk est renseigné même sans trottoir géométrisé
# --------------------------------------------------

import re
import numpy as np
import pandas as pd

# --------------------------------------------------
# Helpers maxspeed et tags CH (optionnel mais recommandé)
# --------------------------------------------------

def parse_maxspeed_kmh(v) -> float | None:
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return None
    if isinstance(v, (list, tuple, set)):
        v = list(v)[0] if len(v) else None
    if v is None:
        return None

    s = str(v).lower().strip()
    s = s.replace("km/h", "").replace("kph", "").strip()

    # valeurs non numériques fréquentes
    if s in {"none", "signals", "variable"}:
        return None

    # ex: "30;50" -> prendre la première valeur
    if ";" in s:
        s = s.split(";")[0].strip()

    m = re.search(r"(\d+(?:\.\d+)?)", s)
    if not m:
        return None
    try:
        return float(m.group(1))
    except Exception:
        return None


def has_explicit_sidewalk(row) -> bool:
    """
    True si OSM indique un trottoir via sidewalk=* ou footway=sidewalk,
    y compris formes left/right/both/yes/separate.
    """
    sw = row.get("sidewalk", None)
    if isinstance(sw, (list, tuple, set)):
        sw = list(sw)[0] if len(sw) else None
    sw = str(sw).lower().strip() if sw not in (None, np.nan) else ""

    # sidewalk=no doit être strictement excluant
    if sw == "no":
        return False

    if sw in {"yes", "both", "left", "right", "separate"}:
        return True

    # certains contributeurs utilisent footway=sidewalk sur l'axe routier
    fw = row.get("footway", None)
    if isinstance(fw, (list, tuple, set)):
        fw = list(fw)[0] if len(fw) else None
    fw = str(fw).lower().strip() if fw not in (None, np.nan) else ""
    if fw == "sidewalk":
        return True

    return False


def is_zone_20(row) -> bool:
    """
    Détecte des encodages CH de zone 20 (si présents).
    Fonction conservatrice: renvoie True si tag explicite zone20.
    """
    # zone:maxspeed=CH:zone20
    z = row.get("zone:maxspeed", None)
    if isinstance(z, (list, tuple, set)):
        z = list(z)[0] if len(z) else None
    z = str(z).strip() if z not in (None, np.nan) else ""
    if z == "CH:zone20":
        return True

    # maxspeed:type=CH:zone20
    mst = row.get("maxspeed:type", None)
    if isinstance(mst, (list, tuple, set)):
        mst = list(mst)[0] if len(mst) else None
    mst = str(mst).strip() if mst not in (None, np.nan) else ""
    if mst == "CH:zone20":
        return True

    return False


# --------------------------------------------------
# Classification marche
# --------------------------------------------------

WALK_DEDICATED_HW = {"footway", "pedestrian", "steps", "corridor", "platform"}
WALK_PATHLIKE_HW = {"path"}

# highways routiers qu'on accepte comme supports proxy quand sidewalk/zone apaisée
WALK_PROXY_ROAD_HW = {
    "motorway", "trunk", "primary", "secondary", "tertiary",
    "motorway_link", "trunk_link", "primary_link", "secondary_link", "tertiary_link",
    "unclassified", "residential", "living_street", "service",
}

def classify_walk(row) -> str:
    hw = str(row.get("highway", "")).lower()
    foot = str(row.get("foot", "")).lower()
    access = str(row.get("access", "")).lower()

    # exclusions strictes uniquement si marche explicitement interdite
    if foot == "no":
        return "exclude"

    # access=no/private excluant sauf si foot override
    if access in {"no", "private"} and foot not in {"yes", "designated", "permissive"}:
        return "exclude"

    # dédié explicite
    if hw in WALK_DEDICATED_HW:
        return "walk_dedicated"

    # path
    if hw in WALK_PATHLIKE_HW:
        return "walk_path"

    # proxy piéton si trottoir explicite sur un axe routier
    if hw in WALK_PROXY_ROAD_HW and has_explicit_sidewalk(row):
        return "walk_proxy_sidewalk"

    # proxy piéton si zone 20 explicite ou maxspeed <= 20
    # (si tu veux uniquement zone 20, commente le test maxspeed)
    if hw in WALK_PROXY_ROAD_HW:
        v = parse_maxspeed_kmh(row.get("maxspeed", None))
        if is_zone_20(row) or (v is not None and v <= 20):
            return "walk_proxy_zone20"

    return "other"


# --------------------------------------------------
# Classification vélo
# --------------------------------------------------

def classify_bike(row) -> str:
    hw = str(row.get("highway", "")).lower()
    bicycle = str(row.get("bicycle", "")).lower()
    access = str(row.get("access", "")).lower()

    if bicycle == "no":
        return "exclude"

    if access in {"no", "private"} and bicycle not in {"yes", "designated", "permissive"}:
        return "exclude"

    if hw == "cycleway":
        return "bike_cycleway"

    cw = str(row.get("cycleway", "")).lower()
    cwl = str(row.get("cycleway:left", "")).lower()
    cwr = str(row.get("cycleway:right", "")).lower()
    cwb = str(row.get("cycleway:both", "")).lower()

    # pistes cyclables séparées
    if any("track" in x or "separated" in x for x in (cw, cwl, cwr, cwb)):
        return "bike_track"

    # bandes cyclables
    if any("lane" in x or "advisory" in x for x in (cw, cwl, cwr, cwb)):
        return "bike_lane"

    # shared
    if any("shared" in x for x in (cw, cwl, cwr, cwb)):
        return "bike_shared"

    if hw in {"path", "track", "bridleway"}:
        if bicycle in {"designated", "yes"}:
            return "bike_path_designated"
        return "bike_path"

    if hw == "pedestrian" and bicycle in {"yes", "designated", "permissive"}:
        return "bike_pedestrian_area"

    motorized_highways = {
        "motorway", "trunk", "primary", "secondary", "tertiary",
        "motorway_link", "trunk_link", "primary_link", "secondary_link", "tertiary_link",
        "unclassified", "residential", "living_street", "service",
    }
    if hw in motorized_highways:
        if hw in {"motorway", "motorway_link"} and bicycle not in {"yes", "designated"}:
            return "exclude"
        if hw in {"trunk", "trunk_link"} and bicycle not in {"yes", "designated", "permissive"}:
            return "exclude"
        return "bike_road"

    if hw == "bus_guideway" and bicycle in {"yes", "designated"}:
        return "bike_busway"

    return "other"


# --------------------------------------------------
# Application aux ways + extraction des sous-jeux
# --------------------------------------------------

ways = ways_raw.copy()

ways["walk_class"] = ways.apply(classify_walk, axis=1)
ways["bike_class"] = ways.apply(classify_bike, axis=1)

# Piéton:
# · dédié = walk_dedicated + walk_path
# · proxy = walk_proxy_sidewalk + walk_proxy_zone20
walk_dedicated = ways[ways["walk_class"].isin({"walk_dedicated", "walk_path"})].copy()
walk_proxy = ways[ways["walk_class"].isin({"walk_proxy_sidewalk", "walk_proxy_zone20"})].copy()

# Vélo:
bike_valid_classes = {
    "bike_cycleway", "bike_track", "bike_lane", "bike_shared",
    "bike_path", "bike_path_designated", "bike_pedestrian_area",
    "bike_road", "bike_busway",
}
bike_base = ways[ways["bike_class"].isin(bike_valid_classes)].copy()

walk_dedicated.shape, walk_proxy.shape, bike_base.shape

## 4 · Consolider la marche (plateformes · traversées · proxy trottoirs)

**Déduplication intelligente** : Supprime les proxy redondants près d'infrastructures dédiées

**3 types de consolidation pour combler les lacunes OSM :**

- Types de voies : `residential`, `living_street`, `service`

**1. Plateformes** (`add_platform_edges=True`)- Filtrage par vitesse (`maxspeed ≤ 30 km/h`)

- Bords de quais/arrêts `public_transport=platform`- Axes routiers locaux comme approximation piétonne

- Espaces piétons souvent absents du réseau highway**3. Proxy trottoirs** (`add_sidewalk_proxy=True`)



**2. Traversées synthétiques** (`add_crossing_edges=True`)- Rattachement aux routes proches

- Points `highway=crossing` → segments perpendiculaires- Largeur configurable (`crossing_halfwidth_m`)

In [ ]:
from shapely.geometry import LineString
import geopandas as gpd
import pandas as pd
import osmnx as ox


def platform_edges_from_polygon(aoi_poly) -> gpd.GeoDataFrame:
    tags = {"public_transport": "platform"}
    out_cols = ["geometry", "walk_class", "source", "osm_id"]

    try:
        plats = ox.features_from_polygon(aoi_poly, tags).reset_index()
        plats = gpd.GeoDataFrame(plats, geometry="geometry", crs="EPSG:4326")

        # ID OSM stable (element/id)
        if "element" in plats.columns and "id" in plats.columns:
            plats["osm_id"] = plats["element"].astype(str) + "/" + plats["id"].astype(str)
        else:
            plats["osm_id"] = None

        plats = plats[plats.geometry.type.isin(["Polygon", "MultiPolygon"])].copy()
        plats["geometry"] = plats.geometry.boundary
        plats = plats.explode(index_parts=False, ignore_index=True)

        plats = plats[plats.geometry.type.isin(["LineString", "MultiLineString"])].copy()
        plats = plats.explode(index_parts=False, ignore_index=True)

        plats["walk_class"] = "platform_edge"
        plats["source"] = "OSM_platform_polygon"

        # garder osm_id + colonnes standard
        return plats[out_cols].copy()

    except Exception:
        return gpd.GeoDataFrame(columns=out_cols, geometry="geometry", crs="EPSG:4326")


def has_sidewalk_tag(v) -> bool:
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return False
    if isinstance(v, (list, tuple, set)):
        v = list(v)[0] if len(v) else None
    if v is None:
        return False
    s = str(v).strip().lower()
    return s not in {"", "no", "none", "separate", "separated"}  # à adapter si besoin


def sidewalk_proxy_from_roads(ways: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    roads = ways[ways["highway"].isin(proxy_include_highways)].copy()
    if roads.empty:
        return gpd.GeoDataFrame(columns=["geometry","walk_class","source","osm_id"], geometry="geometry", crs=ways.crs)

    # signal sidewalk
    if "sidewalk" in roads.columns:
        roads["_has_sidewalk"] = roads["sidewalk"].apply(has_sidewalk_tag)
    else:
        roads["_has_sidewalk"] = False

    # vitesse parsée
    if "maxspeed" in roads.columns:
        roads["_maxspeed_kmh"] = roads["maxspeed"].apply(parse_maxspeed_kmh)
    else:
        roads["_maxspeed_kmh"] = None

    # règle d’inclusion
    # · si sidewalk explicite · inclure même si maxspeed est élevé
    # · sinon · garder la logique vitesse ≤ seuil (si parsable)
    if proxy_maxspeed_kmh is not None:
        m_speed = roads["_maxspeed_kmh"].isna() | (roads["_maxspeed_kmh"] <= float(proxy_maxspeed_kmh))
    else:
        m_speed = pd.Series(True, index=roads.index)

    m_keep = roads["_has_sidewalk"] | m_speed

    # si tu veux exiger un sidewalk explicite, garde seulement ce cas
    if proxy_require_sidewalk_tag:
        m_keep = roads["_has_sidewalk"]

    roads = roads[m_keep].copy()
    roads = roads.drop(columns=["_has_sidewalk","_maxspeed_kmh"], errors="ignore")

    roads["walk_class"] = "sidewalk_proxy_road_axis"
    roads["source"] = "OSM_highway_proxy"

    keep_cols = ["geometry","walk_class","source","highway","osm_id"]
    for c in ["id","element","sidewalk","maxspeed","name","service","access","foot"]:
        if c in roads.columns:
            keep_cols.append(c)

    return roads[keep_cols].copy()


def build_crossing_edges(
    nodes_cross: gpd.GeoDataFrame,
    roads: gpd.GeoDataFrame,
    operation_crs: str,
) -> gpd.GeoDataFrame:
    """Construit des traversées synthétiques. Les arêtes n'ont pas d'osm_id (None)."""
    out_cols = ["geometry", "walk_class", "source", "osm_id"]

    if nodes_cross.empty or roads.empty:
        return gpd.GeoDataFrame(columns=out_cols, geometry="geometry", crs="EPSG:4326")

    nodes = to_metric_crs(nodes_cross, operation_crs)
    r = to_metric_crs(roads, operation_crs)
    r = r[r.geometry.type.isin(["LineString", "MultiLineString"])].copy()
    if r.empty:
        return gpd.GeoDataFrame(columns=out_cols, geometry="geometry", crs="EPSG:4326")

    sindex = r.sindex
    segs = []

    for pt in nodes.geometry:
        if pt is None or pt.is_empty:
            continue

        buf = pt.buffer(float(crossing_snap_dist_m))
        cand_idx = list(sindex.intersection(buf.bounds))
        if not cand_idx:
            continue

        cand = r.iloc[cand_idx].copy()
        cand["dist"] = cand.distance(pt)
        cand = cand[cand["dist"] <= float(crossing_snap_dist_m)].sort_values("dist")
        if cand.empty:
            continue

        road = cand.iloc[0].geometry
        proj_d = road.project(pt)
        proj = road.interpolate(proj_d)

        # approx tangente locale
        delta = 1.0
        d1 = max(proj_d - delta, 0.0)
        d2 = min(proj_d + delta, road.length)
        p1 = road.interpolate(d1)
        p2 = road.interpolate(d2)

        vx = p2.x - p1.x
        vy = p2.y - p1.y
        norm = (vx**2 + vy**2) ** 0.5
        if norm == 0:
            continue
        vx /= norm
        vy /= norm

        # normale
        nx_ = -vy
        ny_ = vx

        hw = float(crossing_halfwidth_m)
        a = (proj.x - nx_ * hw, proj.y - ny_ * hw)
        b = (proj.x + nx_ * hw, proj.y + ny_ * hw)
        segs.append(LineString([a, b]))

    if not segs:
        return gpd.GeoDataFrame(columns=out_cols, geometry="geometry", crs="EPSG:4326")

    gdf = gpd.GeoDataFrame(
        {
            "walk_class": ["crossing_synthetic"] * len(segs),
            "source": ["OSM_crossing_node_proxy"] * len(segs),
            "osm_id": [None] * len(segs),
        },
        geometry=segs,
        crs=operation_crs,
    )

    return gdf.to_crs("EPSG:4326")[out_cols].copy()
    
walk_parts = []

# dédiés
wd = walk_dedicated.copy()
wd["source"] = "OSM_highway_dedicated"

keep_cols = ["geometry","walk_class","source","highway","osm_id"]
for c in ["id","element","footway","sidewalk","name","maxspeed","service","access","foot"]:
    if c in wd.columns:
        keep_cols.append(c)

walk_parts.append(wd[keep_cols])

if add_platform_edges:
    walk_parts.append(platform_edges_from_polygon(aoi_poly))

roads_for_crossings = ways[ways["highway"].isin(crossing_min_road_classes)].copy()

if add_sidewalk_proxy:
    walk_parts.append(sidewalk_proxy_from_roads(ways))

if add_crossing_edges:
    walk_parts.append(build_crossing_edges(nodes_cross, roads_for_crossings, operation_crs))

walk_edges = gpd.GeoDataFrame(pd.concat(walk_parts, ignore_index=True), geometry="geometry", crs="EPSG:4326")
walk_edges = walk_edges[walk_edges.geometry.notna()].copy()
walk_edges.shape

## 5 · Consolider vélo (classification robuste)

**Typologie d'infrastructure cyclable étendue :**
- **Piste cyclable** : `cycleway`, `cycleway=track/separated` (infrastructure séparée)
- **Bande cyclable** : `cycleway=lane/shared_lane` (marquage au sol)
- **Chemin** : `path`, `track`, `bridleway` avec accès vélo
- **Voie spéciale** : Aires piétonnes, voies de bus avec accès vélo
- **Sur chaussée** : Routes partagées avec circulation motorisée

**Gestion avancée des exclusions :**
- **Autoroutes** : Exclues sauf `bicycle=yes` explicite
- **Routes rapides** : `trunk/trunk_link` exclues par défaut
- **Hiérarchie des tags** : `bicycle=*` override `access=no`

**Gestion des sens de circulation :**
- **Analyse OSM** : Tags `oneway:bicycle` et `oneway`
- **3 directions** : `oneway`, `reverse`, `both`
- **Usage** : Préparation pour graphes orientés réalistes

In [ ]:
def map_bike_infra(row) -> str:
    """Mapping final des classes vélo vers typologie simplifiée."""
    cls = str(row.get("bike_class",""))
    
    # Infrastructures séparées de haute qualité
    if cls in {"bike_cycleway", "bike_track"}:
        return "piste_cyclable"
    
    # Voies cyclables marquées
    if cls in {"bike_lane", "bike_shared"}:
        return "bande_cyclable"
    
    # Chemins et sentiers
    if cls in {"bike_path", "bike_path_designated"}:
        return "chemin"
    
    # Aires spéciales avec accès vélo
    if cls in {"bike_pedestrian_area", "bike_busway"}:
        return "voie_spéciale"
    
    # Routes partagées avec circulation
    if cls == "bike_road":
        return "sur_chaussée"
    
    return "inconnu"

def get_bike_direction(row) -> str:
    """
    Détermine le sens de circulation vélo selon la logique OSM.
    Retourne: 'oneway', 'reverse', 'both'
    """
    oneway_bike = str(row.get("oneway:bicycle", "")).lower()
    oneway = str(row.get("oneway", "")).lower()
    
    # Priorité à oneway:bicycle si défini
    if oneway_bike == "yes":
        return "oneway"
    elif oneway_bike == "-1":
        return "reverse"
    elif oneway_bike == "no":
        return "both"
    
    # Sinon, hériter du oneway général
    if oneway == "yes":
        return "oneway"
def get_bike_direction(row) -> str:
    """
    Détermine le sens de circulation vélo selon la logique OSM.
    Retourne: 'oneway', 'reverse', 'both'
    """
    oneway_bike = str(row.get("oneway:bicycle", "")).lower()
    oneway = str(row.get("oneway", "")).lower()
    
    # Priorité à oneway:bicycle si défini
    if oneway_bike == "yes":
        return "oneway"
    elif oneway_bike == "-1":
        return "reverse"
    elif oneway_bike == "no":
        return "both"
    
    # Sinon, hériter du oneway général
    if oneway == "yes":
        return "oneway"
    elif oneway == "-1":
        return "reverse"
    
    # Par défaut: double sens
    return "both"

bike_edges = bike_base.copy()
bike_edges["infra_bike"] = bike_edges.apply(map_bike_infra, axis=1)
bike_edges["bike_direction"] = bike_edges.apply(get_bike_direction, axis=1)
bike_edges["source"] = "OSM_highway_bike"

keep_cols = ["geometry","infra_bike","bike_direction","source","highway"]
for c in ["cycleway","cycleway:left","cycleway:right","bicycle","oneway","oneway:bicycle","name","maxspeed","osmid"]:
    if c in bike_edges.columns:
        keep_cols.append(c)

bike_edges = gpd.GeoDataFrame(bike_edges[keep_cols].copy(), geometry="geometry", crs="EPSG:4326")
bike_edges.shape

## 6 · Nettoyage léger

- Garde les `living_street` et voiries spéciales

**Consolidation finale :**- Paramètres : distance (8m), parallélisme (20°), longueur min (8m)

1. **Conversion métrique** → calculs de distance précis- Supprime les axes routiers redondants près d'infrastructures piétonnes

2. **Éclatement** → segments simples (MultiLineString → LineString)**Déduplication proxy/dédiés :**

3. **Simplification** → réduction points redondants (optionnel)

4. **Longueurs** → attribut `len_m` en mètres- **Proxy_local** : voiries locales (residential/service)

- **Proxy_primary** : axes motorisés (primary/secondary/tertiary)

**Classification piétonne affinée :**- **Crossing** : traversées synthétiques
- **Dedicated** : infrastructures piétonnes pures

In [ ]:
# --------------------------------------------------
# Section 6 · Consolidation finale réseau piéton
# --------------------------------------------------
from shapely.ops import unary_union

# Convertir en CRS métrique pour les opérations
walk_edges_p = to_metric_crs(walk_edges, operation_crs)
bike_edges_p = to_metric_crs(bike_edges, operation_crs)

walk_edges_p = explode_lines(walk_edges_p)
bike_edges_p = explode_lines(bike_edges_p)

walk_edges_p = simplify_gdf(walk_edges_p, simplify_tol_m, operation_crs)
bike_edges_p = simplify_gdf(bike_edges_p, simplify_tol_m, operation_crs)

# Longueurs métriques
walk_edges_p["len_m"] = walk_edges_p.length
bike_edges_p["len_m"] = bike_edges_p.length


# --------------------------------------------------
# walk_role · dedicated | crossing | proxy_primary | proxy_local
# --------------------------------------------------

motor_axes = {
    "primary", "secondary", "tertiary",
    "primary_link", "secondary_link", "tertiary_link",
}

def assign_walk_role(row) -> str:
    wc = str(row.get("walk_class", ""))
    hw = str(row.get("highway", ""))

    if wc == "crossing_synthetic":
        return "crossing"

    if wc == "sidewalk_proxy_road_axis":
        if hw in motor_axes:
            return "proxy_primary"
        return "proxy_local"

    return "dedicated"

walk_edges_p["walk_role"] = walk_edges_p.apply(assign_walk_role, axis=1)



# V2
## -------------------------------------------------
# Supprimer les doublons routes / trottoires
# Principe robuste:
# · ne filtrer QUE les axes structurants (primary/secondary/tertiary + links)
# · ne jamais supprimer residential/unclassified/service/living_street
## --------------------------------------------------

if clean_doublons:
    ded = walk_edges_p[walk_edges_p["walk_class"] != "sidewalk_proxy_road_axis"].copy()
    prox = walk_edges_p[walk_edges_p["walk_class"] == "sidewalk_proxy_road_axis"].copy()

    # axes structurants uniquement = candidats à suppression
    motor_axes = {
        "primary", "secondary", "tertiary",
        "primary_link", "secondary_link", "tertiary_link", "residential", "living_street",
    }

    # tout ce qui n'est pas motor_axes est conservé (incl. residential)
    prox_keep = prox[~prox["highway"].isin(motor_axes)].copy()
    prox_test = prox[prox["highway"].isin(motor_axes)].copy()

    if (not ded.empty) and (not prox_test.empty):
        sidx = ded.sindex


        # paramètres
        buffer_m = 6.0          # distance bord de chaussée
        min_len_m = 8.0
        cover_ratio_drop = 0.5 # preuve forte de doublon
        min_cover_len_m = 20.0  # évite de supprimer sur petits bouts

        keep_flags = []

        for _, row in prox_test.iterrows():
            geom = row.geometry
            if geom is None or geom.is_empty:
                keep_flags.append(True)
                continue

            # candidats dédiés proches (par bounds)
            hits = list(sidx.intersection(geom.buffer(buffer_m).bounds))
            if not hits:
                keep_flags.append(True)
                continue

            near = ded.iloc[hits].copy()
            near = near[near.distance(geom) <= buffer_m].copy()
            if "len_m" in near.columns:
                near = near[near["len_m"] >= min_len_m].copy()
            if near.empty:
                keep_flags.append(True)
                continue

            # zone couverte par dédiés proches
            cover = unary_union(list(near.geometry))
            cover_zone = cover.buffer(buffer_m)

            # mesurer couverture du proxy
            inter = geom.intersection(cover_zone)
            covered_len = inter.length if not inter.is_empty else 0.0
            total_len = geom.length if geom.length else 0.0
            ratio = covered_len / total_len if total_len > 0 else 0.0

            # décision · on supprime seulement si "vraiment doublon" sur une longueur suffisante
            drop = (covered_len >= float(min_cover_len_m)) and (ratio >= float(cover_ratio_drop))
            keep_flags.append(not drop)

        prox_test = prox_test.loc[keep_flags].copy()

    walk_edges_p = pd.concat([ded, prox_keep, prox_test], ignore_index=True)



# --------------------------------------------------
# Déduplication géométrique finale
# --------------------------------------------------
if deduplicate_by_geometry:
    walk_edges_p["__wkb"] = walk_edges_p.geometry.to_wkb()
    walk_edges_p = walk_edges_p.drop_duplicates(subset="__wkb").drop(columns="__wkb")

    bike_edges_p["__wkb"] = bike_edges_p.geometry.to_wkb()
    bike_edges_p = bike_edges_p.drop_duplicates(subset="__wkb").drop(columns="__wkb")

## 7 · Topologie et graphe

### Graphe orienté vs non-orienté pour le vélo

**Graphe non-orienté (traditionnel) :**
- Chaque segment de route = 1 arête bidirectionnelle
- Navigation possible dans les 2 sens sur toutes les arêtes
- Simple mais **ignore les contraintes de circulation réelles**

**Graphe orienté (réaliste) :**
- Respecte les sens de circulation selon les tags OSM :
  - `oneway:bicycle=yes` → sens unique vélo (1 arête u→v)
  - `oneway:bicycle=no` → double sens forcé (2 arêtes u→v + v→u)  
  - `oneway=yes` → sens unique général hérité par le vélo
  - Par défaut → double sens (2 arêtes)

**Implications pratiques :**

1. **Routage réaliste** : Les itinéraires respectent les sens uniques cyclables
2. **Analyse de connectivité** : Identifie les culs-de-sac directionnels
3. **Mesures de centralité** : Calculs précis des flux directionnels
4. **Accessibilité** : Evaluation correcte des zones atteignables

**Cas typiques à Genève :**
- Rues en sens unique avec contre-sens cyclable
- Pistes cyclables unidirectionnelles  
- Zones 30 bidirectionnelles
- Axes motorisés avec aménagements asymétriques

**Usage :**
- `bike_directed_graph = True` : Analyses de mobilité réalistes
- `bike_directed_graph = False` : Analyses topologiques générales

In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
import osmnx as ox
from shapely.geometry import Point, LineString, MultiPoint
from shapely.ops import split
import networkx as nx

def _iter_intersection_points(geom):
    """
    Retourne une liste de Points à partir d'une géométrie d'intersection.
    Ignore les intersections de type LineString (superpositions).
    """
    if geom is None or geom.is_empty:
        return []
    gt = geom.geom_type
    if gt == "Point":
        return [geom]
    if gt == "MultiPoint":
        return list(geom.geoms)
    if gt == "GeometryCollection":
        pts = []
        for g in geom.geoms:
            pts.extend(_iter_intersection_points(g))
        return pts
    return []  # LineString, MultiLineString, Polygon, etc.

def split_edges_at_intersections(
    edges: gpd.GeoDataFrame,
    crs_metric: str,
    min_seg_len_m: float = 1.0,
) -> gpd.GeoDataFrame:
    """
    Split toutes les LineString aux intersections géométriques (y compris T-intersections).
    Retourne un nouveau GeoDataFrame d'arêtes segmentées (attributs dupliqués).
    """
    e = edges.to_crs(crs_metric).copy()
    e = e[e.geometry.notna()].copy()
    e = e[e.geometry.geom_type == "LineString"].copy()
    e = e.reset_index(drop=True)
    e["edge_id"] = np.arange(len(e), dtype=int)

    if e.empty:
        return e

    sidx = e.sindex
    split_pts_by_edge = {i: [] for i in e.index}

    # Détection intersections avec sindex, en évitant les doublons i<j
    for i, geom_i in zip(e.index, e.geometry):
        cand = list(sidx.intersection(geom_i.bounds))
        for j in cand:
            if j <= i:
                continue
            geom_j = e.geometry.iloc[j]
            if not geom_i.intersects(geom_j):
                continue
            inter = geom_i.intersection(geom_j)
            pts = _iter_intersection_points(inter)
            if not pts:
                continue

            # Ajouter à i et j
            split_pts_by_edge[i].extend(pts)
            split_pts_by_edge[j].extend(pts)

    # Split chaque ligne sur ses points
    out_rows = []
    for idx, row in e.iterrows():
        geom = row.geometry
        pts = split_pts_by_edge.get(idx, [])

        # ajouter endpoints pour stabiliser split (optionnel, mais utile)
        coords = list(geom.coords)
        pts.append(Point(coords[0]))
        pts.append(Point(coords[-1]))

        # dédoublonner les points (par WKB)
        if pts:
            wkb_seen = set()
            pts_u = []
            for p in pts:
                w = p.wkb
                if w not in wkb_seen:
                    wkb_seen.add(w)
                    pts_u.append(p)
            pts = pts_u

        if len(pts) <= 2:
            geoms = [geom]
        else:
            try:
                geoms = list(split(geom, MultiPoint(pts)).geoms)
            except Exception:
                geoms = [geom]

        for g in geoms:
            if g is None or g.is_empty:
                continue
            if g.length < float(min_seg_len_m):
                continue
            new_row = row.drop(labels=["geometry"]).to_dict()
            out_rows.append({**new_row, "geometry": g})

    out = gpd.GeoDataFrame(out_rows, geometry="geometry", crs=crs_metric)
    out = out.reset_index(drop=True)
    return out

def build_osmnx_graph_from_segmented_edges(
    edges_seg: gpd.GeoDataFrame,
    crs_metric: str,
    snap_precision_m: float = 0.5,
    directed: bool = False,
):
    e = edges_seg.to_crs(crs_metric).copy()
    e = e[e.geometry.notna()].copy()
    e = e[e.geometry.geom_type == "LineString"].copy()
    e = e.reset_index(drop=True)

    def _round_xy(x, y):
        return (
            round(x / snap_precision_m) * snap_precision_m,
            round(y / snap_precision_m) * snap_precision_m,
        )

    u_xy, v_xy = [], []
    for geom in e.geometry:
        coords = list(geom.coords)
        x1, y1 = coords[0]
        x2, y2 = coords[-1]
        u_xy.append(_round_xy(x1, y1))
        v_xy.append(_round_xy(x2, y2))

    uniq_xy = pd.Index(pd.unique(pd.Index(u_xy + v_xy)))
    nodes = gpd.GeoDataFrame(
        {"osmid": np.arange(len(uniq_xy), dtype=int)},
        geometry=[Point(x, y) for (x, y) in uniq_xy],
        crs=crs_metric,
    )
    nodes["x"] = nodes.geometry.x
    nodes["y"] = nodes.geometry.y
    nodes = nodes.set_index("osmid", drop=True)

    xy_to_id = {xy: int(i) for i, xy in enumerate(uniq_xy)}

    e["u"] = [xy_to_id[xy] for xy in u_xy]
    e["v"] = [xy_to_id[xy] for xy in v_xy]
    e["length"] = e.geometry.length

    # supprimer self-loops dégénérés
    e = e[e["u"] != e["v"]].copy()

    # key unique par (u,v)
    e["_key"] = e.groupby(["u", "v"]).cumcount()

    edges_gdf = e.set_index(["u", "v", "_key"], drop=True)
    edges_gdf.index = edges_gdf.index.set_names(["u", "v", "key"])

    if not nodes.index.is_unique:
        raise ValueError("nodes index not unique after snapping")
    if not edges_gdf.index.is_unique:
        raise ValueError("edges index (u,v,key) not unique, check snapping or duplicates")

    G = ox.graph_from_gdfs(nodes, edges_gdf)

    if not directed:
        G = nx.MultiGraph(G)

    edges_out = edges_gdf.reset_index()

    return G, nodes, edges_out

def build_directed_bike_graph_from_segmented_edges(
    edges_seg: gpd.GeoDataFrame,
    crs_metric: str,
    snap_precision_m: float = 0.5,
):
    """
    Construit un graphe orienté vélo en respectant les sens de circulation.
    """
    e = edges_seg.to_crs(crs_metric).copy()
    e = e[e.geometry.notna()].copy()
    e = e[e.geometry.geom_type == "LineString"].copy()
    e = e.reset_index(drop=True)

    def _round_xy(x, y):
        return (
            round(x / snap_precision_m) * snap_precision_m,
            round(y / snap_precision_m) * snap_precision_m,
        )

    # Expansion des arêtes selon leur direction
    expanded_edges = []
    
    for idx, row in e.iterrows():
        geom = row.geometry
        coords = list(geom.coords)
        x1, y1 = coords[0]
        x2, y2 = coords[-1]
        u_xy = _round_xy(x1, y1)
        v_xy = _round_xy(x2, y2)
        
        direction = row.get('bike_direction', 'both')
        
        # Créer les arêtes selon la direction
        if direction == 'oneway':
            # Sens normal uniquement (u -> v)
            expanded_edges.append({**row.drop('geometry'), 'u_xy': u_xy, 'v_xy': v_xy, 'geometry': geom})
        elif direction == 'reverse':
            # Sens inverse uniquement (v -> u)
            reversed_geom = LineString(coords[::-1])
            expanded_edges.append({**row.drop('geometry'), 'u_xy': v_xy, 'v_xy': u_xy, 'geometry': reversed_geom})
        else:  # 'both'
            # Double sens
            expanded_edges.append({**row.drop('geometry'), 'u_xy': u_xy, 'v_xy': v_xy, 'geometry': geom})
            reversed_geom = LineString(coords[::-1])
            expanded_edges.append({**row.drop('geometry'), 'u_xy': v_xy, 'v_xy': u_xy, 'geometry': reversed_geom})
    
    e_expanded = gpd.GeoDataFrame(expanded_edges, crs=crs_metric)
    
    # Créer les nœuds uniques
    all_xy = list(e_expanded['u_xy']) + list(e_expanded['v_xy'])
    uniq_xy = pd.Index(pd.unique(pd.Index(all_xy)))
    
    nodes = gpd.GeoDataFrame(
        {"osmid": np.arange(len(uniq_xy), dtype=int)},
        geometry=[Point(x, y) for (x, y) in uniq_xy],
        crs=crs_metric,
    )
    nodes["x"] = nodes.geometry.x
    nodes["y"] = nodes.geometry.y
    nodes = nodes.set_index("osmid", drop=True)

    xy_to_id = {xy: int(i) for i, xy in enumerate(uniq_xy)}

    # Assigner u, v aux arêtes expandues
    e_expanded["u"] = [xy_to_id[xy] for xy in e_expanded['u_xy']]
    e_expanded["v"] = [xy_to_id[xy] for xy in e_expanded['v_xy']]
    e_expanded["length"] = e_expanded.geometry.length
    
    # Nettoyer les colonnes temporaires
    e_expanded = e_expanded.drop(columns=['u_xy', 'v_xy'])

    # Supprimer self-loops
    e_expanded = e_expanded[e_expanded["u"] != e_expanded["v"]].copy()

    # Key unique par (u,v)
    e_expanded["_key"] = e_expanded.groupby(["u", "v"]).cumcount()

    edges_gdf = e_expanded.set_index(["u", "v", "_key"], drop=True)
    edges_gdf.index = edges_gdf.index.set_names(["u", "v", "key"])

    # Construire le graphe dirigé
    G = ox.graph_from_gdfs(nodes, edges_gdf)
    # G reste un MultiDiGraph pour le graphe orienté

    edges_out = edges_gdf.reset_index()

    return G, nodes, edges_out

In [ ]:
# 1) segmenter aux intersections
walk_edges_seg = split_edges_at_intersections(
    walk_edges_p,
    crs_metric=operation_crs,
    min_seg_len_m=1.0,
)
print("Segmentation piéton terminée.")

bike_edges_seg = split_edges_at_intersections(
    bike_edges_p,
    crs_metric=operation_crs,
    min_seg_len_m=1.0,
)
print("Segmentation vélo terminée.")

# 2) construire graphe OSMnx
G_walk, walk_nodes_gdf, walk_edges_gdf_graph = build_osmnx_graph_from_segmented_edges(
    walk_edges_seg,
    crs_metric=operation_crs,
    snap_precision_m=0.5,
    directed=False,
)
print("Graphe piéton construit.")

if bike_directed_graph:
    G_bike, bike_nodes_gdf, bike_edges_gdf_graph = build_directed_bike_graph_from_segmented_edges(
        bike_edges_seg,
        crs_metric=operation_crs,
        snap_precision_m=0.5,
    )
    print("Graphe vélo orienté construit.")
else:
    G_bike, bike_nodes_gdf, bike_edges_gdf_graph = build_osmnx_graph_from_segmented_edges(
        bike_edges_seg,
        crs_metric=operation_crs,
        snap_precision_m=0.5,
        directed=False,
    )
    print("Graphe vélo non-orienté construit.")

## 8 · Enrichissements automatiques

**Les enrichissements sont appliqués automatiquement aux données exportées en section 8.**

### Nœuds OSM avec attributs

**Attributs intégrés pour nœuds :**
- **Traversées** : `crossing=*`, `traffic_signals=yes`
- **Barrières** : `barrier=bollard/cycle_barrier`
- **Apaisement** : `traffic_calming=*`
- **Transport** : `public_transport=stop_position`

### Attributs vélo intégrés

**Surface de roulage :**
- **Surface** : `surface=asphalt/gravel/paving_stones` (valeur OSM brute)

### Zones de conflit piéton-vélo

**Détection automatique :**
1. **Intersections géométriques** entre réseaux
2. **Aires partagées** : `highway=pedestrian` + `bicycle=yes`
3. **Chemins mixtes** : `highway=path` sans ségrégation

In [ ]:
def enrich_nodes_with_osm_attributes(nodes_gdf: gpd.GeoDataFrame, aoi_poly, operation_crs: str) -> gpd.GeoDataFrame:
    """
    Enrichit les nœuds de graphe avec les attributs des nœuds OSM proches.
    """
    # Télécharger les nœuds OSM avec attributs intéressants
    osm_node_tags = {
        "crossing": True,
        "traffic_signals": True, 
        "barrier": True,
        "traffic_calming": True,
        "public_transport": True
    }
    
    try:
        osm_nodes = ox.features_from_polygon(aoi_poly, osm_node_tags).reset_index()
        osm_nodes = gpd.GeoDataFrame(osm_nodes, geometry="geometry", crs="EPSG:4326")
        osm_nodes = osm_nodes[osm_nodes.geometry.type == "Point"].copy()
    except Exception:
        print("Aucun nœud OSM avec attributs trouvé")
        return nodes_gdf
    
    if osm_nodes.empty:
        return nodes_gdf
    
    # Convertir en CRS métrique pour calculs de distance
    nodes_metric = to_metric_crs(nodes_gdf, operation_crs)
    osm_nodes_metric = to_metric_crs(osm_nodes, operation_crs)
    
    # Associer par proximité (rayon 20m)
    snap_distance = 20.0
    nodes_enriched = nodes_gdf.copy()
    
    # Initialiser colonnes d'attributs
    for col in ["crossing", "traffic_signals", "barrier", "traffic_calming"]:
        nodes_enriched[col] = None
    
    if not osm_nodes_metric.empty:
        sindex = osm_nodes_metric.sindex
        
        for idx, node in nodes_metric.iterrows():
            candidates = list(sindex.intersection(node.geometry.buffer(snap_distance).bounds))
            if candidates:
                nearby = osm_nodes_metric.iloc[candidates]
                nearby = nearby[nearby.distance(node.geometry) <= snap_distance]
                
                if not nearby.empty:
                    # Prendre le plus proche
                    closest = nearby.loc[nearby.distance(node.geometry).idxmin()]
                    
                    # Copier attributs
                    for col in ["crossing", "traffic_signals", "barrier", "traffic_calming"]:
                        if col in closest and pd.notna(closest[col]):
                            nodes_enriched.loc[idx, col] = str(closest[col])
    
    return nodes_enriched

def enrich_bike_edges_with_surface(bike_edges: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    """
    Enrichit les arêtes vélo avec attributs de surface brute.
    """
    enriched = bike_edges.copy()
    
    # Surface brute (sans score)
    def get_surface(row) -> str:
        surface = row.get("surface", None)
        if pd.isna(surface) or surface is None:
            return None
        return str(surface)
    
    enriched["surface"] = enriched.apply(get_surface, axis=1)
    
    return enriched

def detect_pedestrian_bike_conflicts(walk_edges: gpd.GeoDataFrame, bike_edges: gpd.GeoDataFrame, 
                                   operation_crs: str) -> gpd.GeoDataFrame:
    """
    Détecte les zones de conflit potentiel entre piétons et vélos.
    """
    # Convertir en CRS métrique
    walk_metric = to_metric_crs(walk_edges, operation_crs)
    bike_metric = to_metric_crs(bike_edges, operation_crs)
    
    conflicts = []
    
    # 1. Intersections géométriques directes
    if not walk_metric.empty and not bike_metric.empty:
        walk_sindex = walk_metric.sindex
        
        for idx, bike_edge in bike_metric.iterrows():
            bike_geom = bike_edge.geometry
            if bike_geom is None or bike_geom.is_empty:
                continue
                
            # Chercher intersections avec réseau piéton
            candidates = list(walk_sindex.intersection(bike_geom.bounds))
            
            for walk_idx in candidates:
                walk_geom = walk_metric.iloc[walk_idx].geometry
                
                if bike_geom.intersects(walk_geom):
                    intersection = bike_geom.intersection(walk_geom)
                    
                    if intersection.geom_type == "Point":
                        conflicts.append({
                            "geometry": intersection,
                            "conflict_type": "intersection",
                            "walk_class": walk_metric.iloc[walk_idx].get("walk_class", "unknown"),
                            "bike_infra": bike_edge.get("infra_bike", "unknown"),
                            "severity": "high" if bike_edge.get("infra_bike") == "sur_chaussée" else "medium"
                        })
                    elif intersection.geom_type in ["LineString", "MultiLineString"]:
                        conflicts.append({
                            "geometry": intersection.centroid,
                            "conflict_type": "shared_space",
                            "walk_class": walk_metric.iloc[walk_idx].get("walk_class", "unknown"),
                            "bike_infra": bike_edge.get("infra_bike", "unknown"),
                            "severity": "medium"
                        })
    
    # 2. Aires partagées explicites (pedestrian + bicycle=yes)
    shared_areas = bike_edges[
        (bike_edges.get("highway") == "pedestrian") & 
        (bike_edges.get("bicycle").isin(["yes", "designated"]))
    ].copy()
    
    for idx, area in shared_areas.iterrows():
        if area.geometry is not None:
            conflicts.append({
                "geometry": area.geometry.centroid,
                "conflict_type": "shared_pedestrian_area",
                "walk_class": "pedestrian_area",
                "bike_infra": "voie_spéciale",
                "severity": "low"
            })
    
    if conflicts:
        conflicts_gdf = gpd.GeoDataFrame(conflicts, crs=operation_crs)
        return conflicts_gdf.to_crs("EPSG:4326")
    else:
        return gpd.GeoDataFrame(
            columns=["geometry", "conflict_type", "walk_class", "bike_infra", "severity"],
            geometry="geometry", crs="EPSG:4326"
        )

# Application des enrichissements (conditionnel)
if enable_enrichments:
    print("=== ENRICHISSEMENTS ACTIVÉS ===")
    print("Enrichissement des nœuds avec attributs OSM...")
    walk_nodes_gdf = enrich_nodes_with_osm_attributes(walk_nodes_gdf, aoi_poly, operation_crs)
    bike_nodes_gdf = enrich_nodes_with_osm_attributes(bike_nodes_gdf, aoi_poly, operation_crs)

    print("Enrichissement des arêtes vélo avec surface...")
    bike_edges_gdf_graph = enrich_bike_edges_with_surface(bike_edges_gdf_graph)

    print("Détection des conflits piéton-vélo...")
    conflict_zones = detect_pedestrian_bike_conflicts(walk_edges_gdf_graph, bike_edges_gdf_graph, operation_crs)
    print(f"Conflits détectés: {len(conflict_zones)}")
    
    # Export des zones de conflit si détectées
    if not conflict_zones.empty and export_edges:
        export_gdf(conflict_zones, "pedestrian_bike_conflicts", export_formats, output_dir, export_crs)
        print(f"Zones de conflit exportées: {len(conflict_zones)}")
    
    print("\n=== ENRICHISSEMENTS INTÉGRÉS ===")
    print("Les nœuds et arêtes exportés incluent maintenant:")
    print("- Nœuds: attributs OSM (crossing, traffic_signals, barrier, traffic_calming)")
    print("- Arêtes vélo: surface OSM brute")
    print(f"- Zones de conflit: {len(conflict_zones)} détectées")
else:
    print("=== ENRICHISSEMENTS DÉSACTIVÉS ===")
    print("Export des données de base uniquement (sans attributs OSM)")
    conflict_zones = gpd.GeoDataFrame(
        columns=["geometry", "conflict_type", "walk_class", "bike_infra", "severity"],
        geometry="geometry", crs="EPSG:4326"
    )

## 9 · Export 

**GraphML** : Format NetworkX pour analyses de graphes avancées

**Formats de sortie configurables :**

- **Parquet** : Format optimal (compression, typage, performance)
- **GeoJSON** : Interopérabilité web/JS
- **GPKG** : SIG desktop (QGIS, ArcGIS)

**CRS final** : Export en EPSG:4326 (standard géographique)

**Couches produites  :**
- **Nodes graph** : Nœuds de graphe + attributs OSM (crossing, traffic_signals, barrier, traffic_calming)
- **Edges graph** : Arêtes de graphe + surface OSM pour vélo
- **Crossing nodes** : Points de traversée OSM
- **Conflict zones** : Zones de conflit piéton-vélo (si détectées)

In [ ]:
if export_edges:
    #export_gdf(walk_edges_seg, "walk_edges_segmented", export_formats, output_dir, export_crs)
    export_gdf(walk_edges_gdf_graph, "walk_edges_graph", export_formats, output_dir, export_crs)
    #export_gdf(bike_edges_seg, "bike_edges_segmented", export_formats, output_dir, export_crs)
    export_gdf(bike_edges_gdf_graph, "bike_edges_graph", export_formats, output_dir, export_crs)

if export_nodes:
    export_gdf(walk_nodes_gdf, "walk_nodes_graph", export_formats, output_dir, export_crs)
    export_gdf(bike_nodes_gdf, "bike_nodes_graph", export_formats, output_dir, export_crs)
    if len(nodes_cross) > 0:
        export_gdf(nodes_cross, "walk_crossing_nodes", export_formats, output_dir, export_crs)

if export_graphml:
    ox.save_graphml(G_walk, filepath=str(output_dir / "walk_graph.graphml"))